[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/earthquake/earthquake.ipynb)

# Earthquake Beachballs

This example computes earthquake focal mechanisms, seismic radiation patterns, and beachball diagrams for slipping geological faults, following Aki and Richards (1980). When rock ruptures along a buried fault plane, opposite walls shear past one another and launch compressional P waves and transverse S waves whose first motions arrive around the globe as outward pushes or inward pulls depending on direction. Projecting the lower half of the focal sphere onto the horizontal ground and shading the push quadrants produces the seismological beachball, from which the fault geometry is recovered up to a two-way plane ambiguity.

A fault orientation is modelled as a single rotor composed from strike, dip and rake turns (`strike * dip * rake`), mapping level ground onto the fault plane and the north-facing prime vertical onto the auxiliary plane across the slip. The slip acts on the surrounding medium as a symmetric double-couple moment `auxiliaries * (faults & Point) + faults * (auxiliaries & Point)` whose rotational curl `(Plane ^ moment(Point)).trace(1, 2)` vanishes, while the P-wave first motion along any unit direction reads directly as `moment(direction) & direction` and diagonalizing `moment(Point) & Point` against `Point.dot(Point)` recovers the pressure, null and tension axes.

In Cartesian moment-tensor notation, strike $\phi_s$, dip $\delta$ and rake $\lambda$ expand into trigonometric component formulas $\mathbf{n} = (-\sin\delta\sin\phi_s,\, \sin\delta\cos\phi_s,\, -\cos\delta)$ and $\mathbf{s} = (\cos\lambda\cos\phi_s + \sin\lambda\cos\delta\sin\phi_s,\, \cos\lambda\sin\phi_s - \sin\lambda\cos\delta\cos\phi_s,\, -\sin\lambda\sin\delta)$ to assemble the $3 \times 3$ moment tensor $M_{ij} = M_0(n_i s_j + n_j s_i)$, with rotational equilibrium written via the Levi-Civita contraction $\epsilon_{ijk} M_{jk} = 0$ and far-field P- and S-wave amplitudes written as $\gamma_i M_{ij} \gamma_j$ and $(\delta_{ik} - \gamma_i \gamma_k) M_{kj} \gamma_j$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_states
from examples.mechanics.earthquake import render, scenarios

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Directions are intrinsic antivectors; fault and auxiliary planes are extrinsic vectors.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Moment = ga.gatype((Plane, Point))                                            # Plane+ <- Point-

north, east, down = mv.basis().dual()                                         # [] Point-, Point-, Point-
# Level ground faces up and the prime vertical faces north before the fault rotor turns them.
ground, prime_vertical = -mv.z, mv.x                                          # [] Plane+, Plane+
# Reference bivectors orient the horizontal turn from north toward east and the dip turn from east toward down.
horizontal = east.commutator(north)                                           # [] Bivector+
upright = down.commutator(east)                                               # [] Bivector+

## Fault orientation as a rotor

Seismology specifies a planar fault and its slip direction by three angles: strike gives the compass azimuth of the fault's horizontal trace, dip measures how steeply the fault plunges to the right of the strike, and rake gives the angle of slip within the fault plane. Composing those three rotations into one rotor maps north onto the slip direction and upward onto the fault normal, carrying level ground onto the fault plane and the prime vertical onto the auxiliary plane orthogonal to the slip.

In [ ]:
# Strike-slip, normal, thrust and oblique faults from the shared scenario parameters.
strikes, dips, rakes = scenarios.strikes, scenarios.dips, scenarios.rakes
seismic_moment = scenarios.seismic_moment

In [ ]:
def fault_frame(strikes: np.ndarray, dips: np.ndarray, rakes: np.ndarray) -> Rotor:
    """Three successive turns by `rakes`, `dips` and `strikes` (`[...]`) orient each fault (`[...] Rotor+`), carrying level ground onto the fault plane and north onto the slip direction."""
    # Rake turns north toward -east, dip turns east toward down, and strike turns north toward east.
    rake = (horizontal * (rakes / 2)).exp()                                   # [...] Rotor+
    dip = (upright * (-dips / 2)).exp()                                       # [...] Rotor+
    strike = (horizontal * (-strikes / 2)).exp()                              # [...] Rotor+
    return strike * dip * rake                                                # [...] Rotor+


frames = fault_frame(np.deg2rad(strikes), np.deg2rad(dips), np.deg2rad(rakes))   # [4] Rotor+
faults, auxiliaries = frames >> ground, frames >> prime_vertical                 # [4] Plane+, [4] Plane+

## Double-couple moment of a slip

Shearing opposite walls of a fault pushes rock on one side along the slip and pulls the opposite side back as a single force couple `auxiliaries * (faults & Point)`. Wedging each output plane of a moment map with its input plane and contracting via `(Plane ^ moment(Point)).trace(1, 2)` measures the net rotational torque without invoking the metric, yielding the shear bivector `faults ^ auxiliaries` for a single couple alone. Because an internal rupture cannot impart net angular momentum to the Earth, elastic stress across the auxiliary plane supplies an equal mirrored couple whose sum cancels the curl and leaves the fault plane and auxiliary plane interchangeable.

In [ ]:
def double_couple(faults: Plane, auxiliaries: Plane, moments: float) -> Moment:
    """Adding the fault slip couple and its mirror across `faults` and `auxiliaries` (`[...] Plane+`) forms a torque-free seismic moment (`[...] Plane+ <- Point-`)."""
    return moments * (auxiliaries * (faults & Point) + faults * (auxiliaries & Point))   # [...] Plane+ <- Point-


def curl(moment: Moment) -> Bivector:
    """Wedging each output plane of `moment` with its dual input plane traces out the net rotational couple (`[...] Bivector+`), which vanishes for any symmetric moment."""
    # Tracing wedges the output plane against the input plane without invoking the metric.
    return (Plane ^ moment(Point)).trace(1, 2)                                # [...] Bivector+


# Compare a single unbalanced couple against the symmetric double couple and its plane-swapped twin.
single = auxiliaries * (faults & Point)                                       # [4] Plane+ <- Point-
moment = double_couple(faults, auxiliaries, seismic_moment)                   # [4] Plane+ <- Point-
exchanged = double_couple(auxiliaries, faults, seismic_moment)                # [4] Plane+ <- Point-

## Seismic radiation and beachball diagrams

Along a unit ray direction, pairing `moment(direction) & direction` gives the radial P-wave first motion, positive for an outward compressional push and negative for an inward dilational pull, while dualizing `moment(direction)` with the metric and subtracting the radial projection isolates the transverse S-wave motion. Shading the positive push regions over an equal-area projection of the lower hemisphere draws the beachball, bounded along its zero-amplitude nodal curves by the fault plane and the auxiliary plane.

In [ ]:
# Polar ring and azimuthal spoke counts on the focal sphere.
n_rings, n_spokes = scenarios.n_rings, scenarios.n_spokes

In [ ]:
def sphere_directions(polar: np.ndarray, azimuths: np.ndarray) -> Point:
    """Tilting away from `down` toward `north` by `polar` (`[n_polar]`) and spinning around the vertical by `azimuths` (`[n_azimuths]`) samples a spherical grid of unit directions (`[n_polar, n_azimuths] Point-`)."""
    tilt = (north.commutator(down) * (-polar[:, None] / 2)).exp()             # [n_polar, 1] Rotor+
    spin = (horizontal * (-azimuths[None, :] / 2)).exp()                      # [1, n_azimuths] Rotor+
    return (spin * tilt) >> down                                              # [n_polar, n_azimuths] Point-


def radiation(moment: Moment, directions: Point) -> tuple[Scalar, Point]:
    """Evaluating `moment` along unit `directions` (`[...] Point-`) separates the radial P-wave push (`[...] Scalar+`) from the transverse S-wave motion (`[...] Point-`)."""
    reading = moment(directions)                                              # [...] Plane+
    push = reading & directions                                               # [...] Scalar+
    # Dualizing the reading plane converts force traction into motion before projecting out the radial P component.
    return push, reading.dual() - directions * push


hemisphere = sphere_directions(np.linspace(0.0, np.pi / 2, n_rings), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [n_rings, n_spokes + 1] Point-

In [ ]:
def principal_axes(moment: Moment) -> tuple[Scalar, Point]:
    """Diagonalizing `moment(Point) & Point` against the metric yields the pressure, null and tension eigenvalues (`[..., 3] Scalar+`) and unit axes (`[..., 3] Point-`)."""
    return (moment(Point) & Point).eigh(Point.dot(Point))                     # [..., 3] Scalar+, [..., 3] Point-


values, axes = principal_axes(moment)                                         # [4, 3] Scalar+, [4, 3] Point-
push, shear = radiation(moment[:, None, None], hemisphere)                    # [4, n_rings, n_spokes + 1] Scalar+, [4, n_rings, n_spokes + 1] Point-
save_figure(render.draw_mechanisms(hemisphere, push, axes), "earthquake_beachballs");

From left to right, the four beachballs show strike-slip, normal, thrust and oblique focal mechanisms oriented with north at the top, dark slate shading on compressional push quadrants, sky-blue markers on the pressure axes inside the pull quadrants, and crimson markers on the tension axes inside the push quadrants.

## Nodal plane recovery from principal axes

Diagonalizing the quadratic form `moment(Point) & Point` against the metric orders the principal directions from the inward-pulling pressure axis through the unstressed null intersection line to the outward-pushing tension axis. Bisecting the tension and pressure axes at forty-five degrees recovers the unit normals to both nodal planes, though seismic waves alone cannot distinguish which plane ruptured and which was auxiliary without independent geological evidence such as surface rupture traces or aftershock alignments.

In [ ]:
def fault_planes(moment: Moment) -> tuple[Plane, Plane]:
    """Bisecting the tension and pressure axes of `moment` recovers the two indistinguishable nodal plane pairs (`[..., 2] Plane+`)."""
    # Eigenvalues from `eigh` ascend from pressure through the null axis to tension.
    _, axes = principal_axes(moment)                                          # [..., 3] Point-
    pressure, tension = axes[..., 0], axes[..., 2]                            # [...] Point-, [...] Point-
    # Sum and difference bisect the principal axes at 45 degrees; inverse duality maps normals back to planes.
    first = ((tension + pressure) / np.sqrt(2)).dual_inverse()                # [...] Plane+
    second = ((tension - pressure) / np.sqrt(2)).dual_inverse()               # [...] Plane+
    return stack([first, second], axis=-1), stack([second, first], axis=-1)   # [..., 2] Plane+, [..., 2] Plane+


found, found_auxiliaries = fault_planes(moment)                               # [4, 2] Plane+, [4, 2] Plane+
rebuilt = double_couple(found, found_auxiliaries, seismic_moment)             # [4, 2] Plane+ <- Point-

## Rake rotation on a fixed fault

Holding strike and dip fixed while turning the rake through a full circle rotates the slip vector inside the fault plane, sweeping the mechanism continuously through normal, strike-slip, and thrust regimes as the four P-wave lobes pivot around the fault disc. Rendering all frames takes roughly fifteen seconds on the GPU.

In [ ]:
# Fixed strike and dip in degrees, frame count, and frame duration for the full rake rotation.
sweep_strike, sweep_dip = scenarios.sweep_strike, scenarios.sweep_dip
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def rake_sweep(n_frames: int) -> Iterator[tuple[Point, Point, Point, Scalar, Point, Scalar, Moment]]:
    """Yields `(normal, slip, axes, ball, fault_sphere, lobes, moment)` across `n_frames` evenly spaced rakes over a full turn on a fixed fault plane."""
    sphere = sphere_directions(np.linspace(0.0, np.pi, 2 * n_rings - 1), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [2 * n_rings - 1, n_spokes + 1] Point-
    for rake in np.linspace(-np.pi, np.pi, n_frames, endpoint=False):
        frame = fault_frame(np.deg2rad(sweep_strike), np.deg2rad(sweep_dip), np.array(rake))   # [] Rotor+
        fault, auxiliary = frame >> ground, frame >> prime_vertical           # [] Plane+, [] Plane+
        fault_moment = double_couple(fault, auxiliary, seismic_moment)        # [] Plane+ <- Point-
        _, fault_axes = principal_axes(fault_moment)                          # [3] Point-
        # Dualizing the fault and auxiliary planes gives the unit normal and slip direction for 3D arrows.
        normal, slip = fault.dual(), auxiliary.dual()                         # [] Point-, [] Point-
        # Rotating the sampling sphere with the fault aligns grid lines cleanly with the nodal planes.
        fault_sphere = frame >> sphere                                        # [2 * n_rings - 1, n_spokes + 1] Point-
        ball, _ = radiation(fault_moment, hemisphere)                         # [n_rings, n_spokes + 1] Scalar+
        lobes, _ = radiation(fault_moment, fault_sphere)                      # [2 * n_rings - 1, n_spokes + 1] Scalar+
        yield normal, slip, fault_axes, ball, fault_sphere, lobes, fault_moment


sweep_states = show_states(
    rake_sweep(n_frames), render.animate_sweep, hemisphere,
    name="earthquake_rake", duration_ms=frame_duration_ms, n_frames=n_frames,
);

On the left, amber arrows trace the opposing slip of the two fault blocks across the translucent amber fault disc while two dark compressional P-wave lobes and two pale dilational lobes rotate around it; on the right, one nodal arc stays pinned to the fixed fault plane while the auxiliary nodal arc and principal axes sweep across the beachball.

In [ ]:
# checks
# Pure shear slip neither dilates nor compresses the rock and leaves the null axis unstressed, so eigenvalues are (-M_0, 0, M_0).
expected_values = np.array([-seismic_moment, 0.0, seismic_moment])
assert ((values - expected_values).abs() < 1e-12).all()
assert ((moment(Point) & Point).det(Point.dot(Point)).abs() < 1e-12).all()
# A single couple exerts net shear torque `faults ^ auxiliaries`, whereas the double couple is curl-free.
assert ((curl(single) - (faults ^ auxiliaries)).norm() < 1e-12).all()
assert (curl(moment).norm() < 1e-12).all()
# S-wave motion is everywhere orthogonal to the ray direction.
assert (shear.dot(hemisphere).abs() < 1e-12).all()
# Exchanging the fault and auxiliary planes leaves the moment unchanged, and bisecting P and T axes recovers both planes.
assert ((exchanged - moment).svdvals() < 1e-12).all()
assert ((rebuilt - moment[:, None]).svdvals() < 1e-12).all()
assert ((found.dot(faults[:, None]).abs().max(axis=-1) - 1.0).abs() < 1e-12).all()
# Across the streamed rake sweep, the fault normal stays fixed, slip stays in the fault plane, and half a turn flips the moment.
sweep_normals = stack([state[0] for state in sweep_states])                   # [n_frames] Point-
sweep_slips = stack([state[1] for state in sweep_states])                     # [n_frames] Point-
sweep_moments = stack([state[6] for state in sweep_states])                   # [n_frames] Plane+ <- Point-
assert ((sweep_normals - sweep_normals[0]).norm() < 1e-12).all()
assert (sweep_normals.dot(sweep_slips).abs() < 1e-12).all()
assert ((sweep_moments[n_frames // 2:] + sweep_moments[:n_frames // 2]).svdvals() < 1e-12).all()